In [1]:
import pathlib

import yaml
import pandas as pd

import jax.random as jr

from torch.utils.data import DataLoader
import hdbscan

import altair as alt
import plotly.express as px
import plotly.graph_objects as go

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core
import trapallada.plot

# import ipywidgets as widgets

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

# Input Files

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [3]:
# input_files_stem = '../results/5_sources/good'
# input_files_stem = '../results/4_sources/3000_epochs'
input_files_stem = '../results/epochs/20000/4_sources/epochs'

It is turned into a `Pathlib`

In [ ]:
input_files_stem = pathlib.Path(input_files_stem)
input_files_stem

In [5]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [6]:
# w_file_upload.value

## Model

From the above *stem*, several file names are derived

In [ ]:
if input_files_stem.is_dir():
    parameters_file = input_files_stem / 'extended_parameters.yaml'
    weights_file = input_files_stem / 'model.eqx'
    results_file = input_files_stem / 'in_training_metrics.parquet'
    print('a directory...')
else:
    parameters_file = input_files_stem.with_suffix('.yaml')
    weights_file = input_files_stem.with_suffix('.eqx')
    results_file = input_files_stem.with_suffix('.parquet')

assert parameters_file.exists()
assert weights_file.exists()
assert results_file.exists()

In [ ]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
# %pycat {parameters_file}
%cat {parameters_file}

## Data

The files making up the dataset used to train the model...

In [9]:
signals_file = pathlib.Path.home() / parameters['dataset']['files']['signals_file']
metadata_file = pathlib.Path.home() / parameters['dataset']['files']['metadata_file']

...are loaded...

In [ ]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
# len(ds)

...along with the metrics from the training process

In [ ]:
results_df = pd.read_parquet(results_file)
results_df

There is an extra data file with information extracted from the *headers* of the FITs files

In [ ]:
headers_file = signals_file.with_name('headers.parquet')
positions_df = pd.read_parquet(headers_file)
positions_df

# Training curves

In [ ]:
def metric_chart(df: pd.DataFrame, metric: str, n_rows: int = 4_000):

    sub_df = df.loc[:n_rows,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

In [14]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [ ]:
sum(trapallada.util.get_layers_n_parameters(model).values())

Data loaders for both the training and validation sets

In [16]:
(train_dl, validation_dl), (train_ds, validation_ds) = trapallada.data.dataloaders_from_dataset(
    ds,
    splitting_parameters=parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    batch_size=parameters['training']['batch_size'], batching_random_state=parameters['PRNG seeds']['training dataloader'], return_datasets=True)

A batch from the **validation** set

In [ ]:
# iter_dl = iter(train_dl) # training
iter_dl = iter(validation_dl) # validation
next(iter_dl), next(iter_dl)
batch = next(iter_dl)
batch[1].shape

What are the different labels, *if any* (useful to check whether every class is present)?

In [ ]:
batch[2].unique()

An element from the above batch is encoded, decoded, and a prediction plotted

In [ ]:
# i = 0
# i = 1
i = 31

# indexes from the above batch
i_s, *_ = batch

def plot_prediction(ds, i):
    
    x = ds[i][1].numpy()
    x_pred, z_mean, z_sd = model(x, jr.PRNGKey(42))
    display(ds.metadata_df.loc[i])
    df = pd.DataFrame({'prediction': x_pred[0], 'actual': x.ravel()})
    n_rows = 2000
    
    return trapallada.plot.df(df.iloc[:n_rows])

# plot_prediction(ds, 25)
plot_prediction(ds, i_s[i].item())

## Javi's paper

A function that looks up a particular source (a `FTZ` file) and plots the requested window along with its reconstruction. It returns a `Series` **relative to the full dataset**.

In [20]:
def plot_source_window(source: str, i_window: int = -1):

    df = ds.metadata_df[ds.metadata_df['source'] == source]

    # display((df['end'] - df['start']).tail())

    try:
        series = df.iloc[i_window]
    
    except IndexError:

        print('not found')

        return None
    
    index = series.name

    display(plot_prediction(ds, index))
    
    # print(f'seen during training? {index in train_ds[:][0]}' )

    return series

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8001.FTZ', -1)
javi_meta = [javi_obs]

In [ ]:
# javi_meta_2 = plot_source_window('P0605950101PNS003SRCTSR8003.FTZ')
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8003.FTZ')
javi_meta.append(javi_obs)

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8005.FTZ')
javi_meta.append(javi_obs)

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8007.FTZ')
javi_meta.append(javi_obs)

In [ ]:
i_javi = [e.name for e in javi_meta if e is not None]
i_javi

# Latent space projection

## 2D

(Out of the full dataset), the curves in the given `DataLoader` are encoded into the latent space (last columns in the resulting `DataFrame`)

In [ ]:
# fulldataset_dl = DataLoader(ds, batch_size=parameters['training']['batch_size'], shuffle=False)
fulldataset_dl = DataLoader(ds, batch_size=128, shuffle=False)
proj_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder)
proj_df.loc[train_ds[:][0], 'set'] = 'training'
proj_df.loc[validation_ds[:][0], 'set'] = 'validation'
proj_df

Embedded space is close to a standard Gaussian?

In [ ]:
proj_df.filter(like='z_').agg(['mean', 'std'])

### Clustering

Clustering using [HDBSCAN](https://github.com/scikit-learn-contrib/hdbscan) with pretty much every default parameter. See [this notebook](https://nbviewer.org/github/scikit-learn-contrib/hdbscan/blob/master/notebooks/How%20HDBSCAN%20Works.ipynb) for an explanation.

In [28]:
def columns_clusters(like: str, min_cluster_size: int) -> pd.Series:

    selected_cols = proj_df.filter(like=like)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size)
    
    labels = clusterer.fit_predict(selected_cols)
    
    print(f'number of clusters: {len(np.unique(labels))}')
    
    return pd.Categorical(labels)

In [ ]:
cluster_in_z = 'cluster in z'
proj_df[cluster_in_z] = columns_clusters('z_', 10)

The number of samples in every cluster

In [ ]:
n_samples_per_cluster = proj_df[cluster_in_z].value_counts()
n_samples_per_cluster

An array containing the indexes of the samples (signals) that do not belong to the large cluster (which is, presumably, noise).

In [ ]:
i_minority = proj_df[proj_df[cluster_in_z] != n_samples_per_cluster.idxmax()].index
len(i_minority)

In [ ]:
cluster_in_projected_space = 'cluster in projected space'
proj_df[cluster_in_projected_space] = columns_clusters('tsne_', 80)

### Extracting positions

There is an extra data file with information extracted from the *headers* of the FITs files

In [ ]:
headers_file = signals_file.with_name('headers.parquet')
positions_df = pd.read_parquet(headers_file)
positions_df

For the sake of convenience

In [93]:
pos_columns = ['RA', 'DEC', 'CRA', 'CDEC']
tsne_columns = [f'tsne_{i}' for i in range(1, 3)]

*Projections* `DataFrame` with positions. Notice the indexes in `proj_df` are kept.

In [ ]:
# pos_and_proj_df = proj_df.join(positions_df, on=['tar file', 'source'], how='left')[pos_columns + tsne_columns]
proj_df_with_positions = proj_df.join(positions_df, on=['tar file', 'source'], how='left')
assert not proj_df_with_positions.isna().any().any()
proj_df_with_positions

In [ ]:
# pos_and_proj_df = positions_df.join(proj_df.set_index(['tar file', 'source']).loc[:, tsne_columns], how='inner')
# pos_and_proj_df

The above `DataFrame` is stored to be used in a query to a database (notice that, actually, `positions_df` would be enough)

In [126]:
# pos_and_proj_df.to_csv('pos_and_proj.csv', index=False)
proj_df_with_positions[pos_columns + tsne_columns].to_csv('pos_and_proj.csv', index=False)

### SIMBAD (XMATCH) database

In [127]:
pd.set_option('display.max_columns', None)

In [ ]:
catalog_file = pathlib.Path('1727127553869A.csv')
assert catalog_file.exists()
catalog_df = pd.read_csv(catalog_file)
catalog_df

In [129]:
# catalog_file = pathlib.Path('Cross_Simbad_r3.5.xlsx')
# assert catalog_file.exists()
# catalog_df = pd.read_excel(catalog_file)
# catalog_df

In [130]:
# catalog_file = pathlib.Path('Cross_Simbad_r3.5.dat')
# assert catalog_file.exists()
# catalog_df = pd.read_csv(catalog_file, delim_whitespace=True)
# catalog_df

The number of *records* in the catalog is much larger than that requested

In [ ]:
len(catalog_df), len(proj_df_with_positions)

These should have `float` type

In [ ]:
catalog_df[pos_columns].dtypes

The number of *matches* for every position

In [ ]:
groupby_catalog = catalog_df.groupby(pos_columns)
groupby_catalog.size()

Out of every group, we keep only the first element (in the given order)

In [ ]:
first_match_catalog_df = groupby_catalog.first()
first_match_catalog_df

The number of positions requested is actually bigger (check the number of groups above)

In [ ]:
len(positions_df)

In [ ]:
_, sample_group = next(iter(groupby_catalog))
sample_group[['main_type', 'other_types']]

In [ ]:
len(catalog_df['main_type'].unique())

Data used for training is matched against the catalog

In [ ]:
match_columns_of_interes = ['main_type', 'other_types', 'main_id'] + pos_columns
# ext_proj_df = pd.merge(
#     positions_df.join(proj_df.set_index(['tar file', 'source']), how='inner').reset_index(), # proj_df enhanced with positions
#     # catalog_df[match_columns_of_interes],  # a few selected columns in the above DataFrame
#     first_match_catalog_df.reset_index()[match_columns_of_interes],  # a few selected columns in the above DataFrame
#     on=pos_columns, how='left'
#     )
ext_proj_df = pd.merge(
    proj_df_with_positions,
    first_match_catalog_df.reset_index()[match_columns_of_interes], # a few columns in the catalog DataFrame
    on=pos_columns, how='left'
)
ext_proj_df

Distinct *types* of objects

In [ ]:
ext_proj_df['main_type'].unique()

In [ ]:
main_type_is_na = ext_proj_df['main_type'].isna()
main_type_is_na.sum()

Only the *identified* light curves are kept

In [166]:
# ext_proj_df.dropna(subset=['main_type'], inplace=True)

### Javi's paper

The clustering (either *in projected space* or *in z*) to be analyzed

In [168]:
# cluster = cluster_in_z
cluster = cluster_in_projected_space

In [169]:
# proj_df.loc[i_javi, ['tsne_1', 'tsne_2', 'set', cluster]]

In [ ]:
ext_proj_df.loc[i_javi, ['tsne_1', 'tsne_2', 'set', cluster, 'main_type']]

From [this issue](https://github.com/vega/altair/pull/3119)

In [199]:
# hover_data=['index', 'source', cluster]
# brush = alt.selection_interval(name="interval")
# alt.data_transformers.enable("vegafusion")
# # alt_fig = alt.Chart(proj_df.reset_index()).mark_point(opacity=0.5).encode(
# alt_fig = alt.Chart(ext_proj_df.reset_index()).mark_point(opacity=0.5).encode(
#     x='tsne_1',
#     y='tsne_2',
#     # color='tar file',
#     color='main_type',
#     # shape='set',
#     tooltip=hover_data
# ).properties(
#     width=1200,
#     height=800
# ).add_params(brush).interactive()
# # jchart = alt.JupyterChart(alt_fig)
# # jchart
# alt_fig


A `DataFrame` with the points in the selection

In [68]:
# selection = (proj_df['tsne_1'] > jchart.selections.interval.value['tsne_1'][0])\
#  & (proj_df['tsne_1'] < jchart.selections.interval.value['tsne_1'][1])\
# & (proj_df['tsne_2'] > jchart.selections.interval.value['tsne_2'][0])\
# & (proj_df['tsne_2'] < jchart.selections.interval.value['tsne_2'][1])
# selection_df = proj_df[selection]
# selection_df.head()

In [69]:
# selection_df[cluster].value_counts()

In [70]:
# hover_data=['index', 'source', cluster]
# # fig = px.scatter(ext_proj_df.reset_index(), x='tsne_1', y='tsne_2', color='main_type', hover_data=hover_data, opacity=0.5, width=1_200, height=800)
# fig = px.scatter(proj_df.reset_index(), x='tsne_1', y='tsne_2', color='tar file', symbol='set', hover_data=hover_data, opacity=0.5, width=1_200, height=800)
# # fig = px.scatter(proj_df.reset_index(), x='tsne_1', y='tsne_2', color=cluster, hover_data=hover_data, opacity=0.5, width=1_200, height=800)
# fig.add_trace(go.Scatter(
#     x=proj_df.loc[i_javi, 'tsne_1'].tolist(), y=proj_df.loc[i_javi, 'tsne_2'].tolist(), mode='markers', marker=dict(size=20, symbol='star', color='blue')
# ))
# fig.show()

In [178]:
ext_proj_df['main_type'].fillna('UNKNOWN', inplace=True)

In [ ]:
ext_proj_df['main_type'].unique()

In [ ]:
selected_main_type = ext_proj_df['main_type'].isin(['OrionV*', 'Low-Mass*', 'LowMassXBin', 'TTauri*'])
selected_main_type

In [ ]:
hover_data=['index', 'source', cluster, 'main_type']
fig = px.scatter(ext_proj_df.reset_index(), x='tsne_1', y='tsne_2', color='main_type', hover_data=hover_data, opacity=0.5, width=1_200, height=800)
fig.add_trace(go.Scatter(
    x=ext_proj_df.loc[i_javi, 'tsne_1'].tolist(), y=proj_df.loc[i_javi, 'tsne_2'].tolist(), mode='markers', marker=dict(size=20, symbol='star', color='blue')
))
# fig = px.scatter(ext_proj_df[selected_main_type].reset_index(), x='tsne_1', y='tsne_2', color='main_type', hover_data=hover_data, opacity=0.5, width=1_200, height=800)
fig.show()

In [ ]:
# i_study = [50301, 38810, 536, 55343]
# i_study = [20563, 27821, 55715]
# i_study = [20610, 37805, 49497]
# i_study = [19815]
# i_study = [31116]
i_study = [30460]
for i in i_study:
    display(plot_prediction(ds, i))

In [ ]:
ext_proj_df.loc[i_study]

In [ ]:
# ds[2239]
# plot_prediction(ds, 2239)
# plot_prediction(ds, 12348)
# plot_prediction(ds, 10480)
# plot_prediction(ds, 5778)
# plot_prediction(ds, 3995)
# plot_prediction(ds, 7102)
# plot_prediction(ds, 10190)
# ------------- (by looking at above 3D picture)
# plot_prediction(ds, 33170)
# plot_prediction(ds, 10868)
# -------------
# plot_prediction(ds, 54087)
# plot_prediction(ds, 36097)
# plot_prediction(ds, 2673)
# plot_prediction(ds, 6004)
# plot_prediction(ds, 16900)
# plot_prediction(ds, 27243)
# plot_prediction(ds, 50287)
# plot_prediction(ds, 49489)
# plot_prediction(ds, 11717) # <---- noise (?)
# plot_prediction(ds, 16903)
# plot_prediction(ds, 12455)
# plot_prediction(ds, 23774)
# plot_prediction(ds, 10696)
# plot_prediction(ds, 22916)
# plot_prediction(ds, 54475)
# -------------
plot_prediction(ds, i_minority[30])

## 3D

The latent space is projected into a 3D space

In [ ]:
proj3D_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder, n_projected_components=3)
proj3D_df.loc[train_ds[:][0], 'set'] = 'training'
proj3D_df.loc[validation_ds[:][0], 'set'] = 'validation'
proj3D_df.head()

In [ ]:
proj3D_df.loc[i_javi].filter(like='tsne_')

In [ ]:
fig = px.scatter_3d(proj3D_df.reset_index(), x='tsne_1', y='tsne_2', z='tsne_3', hover_data=['index'], color='tar file', opacity=0.5, width=1_200, height=800)
fig.add_trace(go.Scatter3d(
    x=proj3D_df.loc[i_javi, 'tsne_1'].tolist(), y=proj3D_df.loc[i_javi, 'tsne_2'].tolist(), z=proj3D_df.loc[i_javi, 'tsne_3'].tolist(), mode='markers', marker=dict(size=12, symbol='diamond', color='blue')
))
fig.show()